In [1]:
import pandas as pd
import lseg.data as ld

from pathlib import Path
from datetime import date

In [2]:
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [3]:
ld.open_session()

<lseg.data.session.Definition object at 0x121e35e80 {name='workspace'}>

# Fecthing Data for Equity Options


The option chains follow the given naming logic: `0#<RIC-root>*.<exchange>`, where `RIC-root` is the company abbreviation (i.e. `MUVGn` for Münchener Rückversicherung) and `exchange` identifies the place of the stock exchange at hand (i.e. `DE` for Germany).

In [4]:
today = date.today()

START_DATE = today.strftime('%Y-%m-%d')
#START_DATE = '2026-05-09'
AS_OF_DATE = pd.to_datetime(START_DATE).normalize()

DATA_DIR = Path('./Option Data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

MAX_MATURITY_YEARS = 1.5
UNDERLYING_PRICE_LOOKBACK_DAYS = 10

OPTION_FIELDS = ['PUTCALLIND', 'PUT_CALL', 'BID', 'ASK', 'EXPIR_DATE', 'STRIKE_PRC', 'CF_CLOSE', 'VEGA']
UNDERLYING_FIELD = 'TRDPRC_1' #'TRDPRC_1'

tickers = ['ROPC.S',    #Roche
           'NOVN.S',    #Novartis
           'NESN.S',    #Nestle
           'ABBN.S',    #ABB
           'UBSG.S',    #UBS
           'CFR.S',     #Richemont
           'ZURN.S',    #Zurich Insurance Group
           'HOLN.S',    #Holcim
           'SRENH.S',   #Swiss Re
           'SCMN.S',    #Swisscom
]

RICs = ['0#ROGE*.EX',
        '0#NOVE*.EX', 
        '0#NESE*.EX',
        '0#ABBE*.EX', 
        '0#UBSE*.EX',
        '0#CFRE*.EX',
        '0#ZURE*.EX',
        '0#HOLE*.EX',
        '0#SREE*.EX',
        '0#SCME*.EX',
]

In [10]:
output_file_list = []
call_price_dfs = []

for i, RIC in enumerate(RICs):
    print(tickers[i], RIC)
    df = ld.get_data(universe=RIC, fields=OPTION_FIELDS)
    df = df.drop(index=df.index[0]).dropna()

    call_price_df = df[df['PUTCALLIND'].str.strip().eq('CALL')].copy()
    call_price_df['price'] = (call_price_df['BID']+call_price_df['ASK'])/2
    call_price_df = call_price_df.rename(columns={'STRIKE_PRC': 'strike', 'EXPIR_DATE': 'expiry_date', 'PUTCALLIND': 'option_type', 'VEGA': 'vega'})
    call_price_df['time_to_maturity_years'] = (pd.to_datetime(call_price_df['expiry_date'])-AS_OF_DATE)/pd.Timedelta(days=365)
    call_price_df = call_price_df[call_price_df['time_to_maturity_years'] <= MAX_MATURITY_YEARS]

    start_lookup_date = (AS_OF_DATE - pd.Timedelta(days=UNDERLYING_PRICE_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    end_lookup_date = AS_OF_DATE.strftime('%Y-%m-%d')
    
    close_price_stock = ld.get_history(universe=tickers[i], interval='1min', count=1, fields=UNDERLYING_FIELD)
    close_price_stock = float(pd.to_numeric(close_price_stock[UNDERLYING_FIELD], errors='coerce').dropna().iloc[-1])

    call_price_df['S0'] = round(close_price_stock, 5)
    call_price_df['START_DATE'] = START_DATE

    output_file = DATA_DIR/f'{RIC}.xlsx'
    call_price_df.to_excel(output_file, index=False)

    call_price_dfs.append(call_price_df)
    output_file_list.append(str(output_file))

ROPC.S 0#ROGE*.EX
NOVN.S 0#NOVE*.EX
NESN.S 0#NESE*.EX
ABBN.S 0#ABBE*.EX
UBSG.S 0#UBSE*.EX
CFR.S 0#CFRE*.EX
ZURN.S 0#ZURE*.EX
HOLN.S 0#HOLE*.EX
SRENH.S 0#SREE*.EX
SCMN.S 0#SCME*.EX


In [11]:
output_file_list

['Stock Data/0#ROGE*.EX.xlsx',
 'Stock Data/0#NOVE*.EX.xlsx',
 'Stock Data/0#NESE*.EX.xlsx',
 'Stock Data/0#ABBE*.EX.xlsx',
 'Stock Data/0#UBSE*.EX.xlsx',
 'Stock Data/0#CFRE*.EX.xlsx',
 'Stock Data/0#ZURE*.EX.xlsx',
 'Stock Data/0#HOLE*.EX.xlsx',
 'Stock Data/0#SREE*.EX.xlsx',
 'Stock Data/0#SCME*.EX.xlsx']

In [12]:
for i, df in enumerate(call_price_dfs):
    print('=' * 80)
    print(f'Overview for dataframe {i}')
    print('=' * 80)

    valuation_date = df['valuation_date'].iloc[0] if 'valuation_date' in df.columns else AS_OF_DATE.date()
    s0 = df['S0'].iloc[0]

    print(f'Underlying:      {tickers[i]}')
    print(f'Valuation date:  {valuation_date}')
    print(f'S0:              {s0}')
    print(f'Total samples:   {len(df)}')
    print()

    overview = df.groupby('expiry_date').size().reset_index(name='samples')
    overview['time_to_maturity_days'] = (pd.to_datetime(overview['expiry_date']) - pd.to_datetime(valuation_date)).dt.days
    overview = overview[['expiry_date', 'time_to_maturity_days', 'samples']]

    print(overview.to_string(index=False))
    print()

Overview for dataframe 0
Underlying:      ROPC.S
Valuation date:  2026-05-13
S0:              318.6
Total samples:   236

expiry_date  time_to_maturity_days  samples
 2026-05-15                      2       22
 2026-06-19                     37       46
 2026-07-17                     65       29
 2026-09-18                    128       43
 2026-12-18                    219       50
 2027-03-19                    310       26
 2027-06-18                    401       20

Overview for dataframe 1
Underlying:      NOVN.S
Valuation date:  2026-05-13
S0:              116.16
Total samples:   204

expiry_date  time_to_maturity_days  samples
 2026-05-15                      2       20
 2026-06-19                     37       40
 2026-07-17                     65       29
 2026-09-18                    128       30
 2026-12-18                    219       40
 2027-03-19                    310       26
 2027-06-18                    401       19

Overview for dataframe 2
Underlying:      NESN.S


# Fetching Stocks data

In [13]:
tickers = ['ROPC.S',    #Roche
           'NOVN.S',    #Novartis
           'NESN.S',    #Nestle
           'ABBN.S',    #ABB
           'UBSG.S',    #UBS
           'CFR.S',     #Richemont
           'ZURN.S',    #Zurich Insurance Group
           'HOLN.S',    #Holcim
           'SRENH.S',   #Swiss Re
           'SCMN.S',    #Swisscom
]

DIVIDEND_FIELDS = [
    'TR.DivDate',
    'TR.DivExDate',
    'TR.DivRecordDate',
    'TR.DivPayDate',
    'TR.DivType',
    'TR.DivPaymentType',
    'TR.DivCurr',
    'TR.DivUnadjustedGross',
    'TR.DivAdjustedGross',
]

DATA_DIR = Path('./Stock Data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

last_year = today.replace(year=today.year - 2)
LAST_YEAR = last_year.strftime('%Y-%m-%d')
# LAST_YEAR = '2024-05-09'
END_DATE = START_DATE

In [14]:
ticker_dfs = []
dividend_dfs = []

for ticker in tickers:
    ticker_df = ld.get_history(universe=ticker, interval='daily', start=LAST_YEAR, end=START_DATE, fields=UNDERLYING_FIELD)

    dividend_df = ld.get_data(universe=ticker, fields=DIVIDEND_FIELDS, parameters={'SDate': LAST_YEAR, 'EDate': START_DATE, 'DateType': 'ED'})
    filename_df = DATA_DIR / f'{ticker}.xlsx'

    with pd.ExcelWriter(filename_df, engine='openpyxl') as writer:
        ticker_df.to_excel(writer, sheet_name='Stock Data')
        dividend_df.to_excel(writer, sheet_name='Dividends', index=False)

    ticker_dfs.append(ticker_df)
    dividend_dfs.append(dividend_df)